# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


The whole `matrix` class is defined in the next cell (initialisation, indexing, assignment, the methods from part 2, the arithmetic from part 4 and the operator overloads from part 5). Each part is tested in its own cell below.

In [1]:
class matrix:
    """A simple matrix stored as a list of lists."""

    def __init__(self, *args):
        if len(args) == 2 and all(isinstance(a, int) and not isinstance(a, bool) for a in args):
            n, m = args
            if n < 1 or m < 1:
                raise ValueError("matrix dimensions must be positive")
            self.data = [[0] * m for _ in range(n)]
        elif len(args) == 1 and isinstance(args[0], (list, matrix)):
            self.data = self._validated_copy(args[0])
        else:
            raise TypeError("use matrix(n, m) or matrix(list_of_lists)")

    @staticmethod
    def _is_number(v):
        return isinstance(v, (int, float)) and not isinstance(v, bool)

    @staticmethod
    def _validated_copy(src):
        """Check that src is a proper rectangular list of lists of numbers; return a copy."""
        if isinstance(src, matrix):
            return [row[:] for row in src.data]
        if not isinstance(src, list) or len(src) == 0:
            raise ValueError("matrix must be a non-empty list of rows")
        for i, row in enumerate(src):
            if not isinstance(row, list):
                raise TypeError(f"row {i} is not a list")
        width = len(src[0])
        if width == 0:
            raise ValueError("rows must be non-empty")
        for i, row in enumerate(src):
            if len(row) != width:
                raise ValueError(f"row {i} has {len(row)} entries, expected {width}")
            for v in row:
                if not matrix._is_number(v):
                    raise TypeError(f"non-numeric entry {v!r} in row {i}")
        return [row[:] for row in src]

    def __getitem__(self, key):
        if isinstance(key, tuple):
            if len(key) != 2:
                raise IndexError("use M[i, j]")
            r, c = key
            if isinstance(r, int) and isinstance(c, int):
                return self.data[r][c]
            rows = [self.data[r]] if isinstance(r, int) else self.data[r]
            return matrix([[row[c]] if isinstance(c, int) else row[c] for row in rows])
        if isinstance(key, int):
            return self.data[key]
        if isinstance(key, slice):
            return matrix(self.data[key])
        raise TypeError("invalid index")

    def __setitem__(self, key, value):
        if isinstance(key, tuple) and len(key) == 2:
            r, c = key
            if isinstance(r, int) and isinstance(c, int):
                if not self._is_number(value):
                    raise TypeError("entries must be numbers")
                self.data[r][c] = value
                return
            if r == slice(None) and c == slice(None):
                self.assign(value)
                return
        elif isinstance(key, int):
            if not (isinstance(value, list) and len(value) == self.shape()[1]
                    and all(self._is_number(v) for v in value)):
                raise ValueError("row must be a list of numbers with the right length")
            self.data[key] = value[:]
            return
        elif key == slice(None):
            self.assign(value)
            return
        raise TypeError("unsupported assignment")

    def assign(self, other):
        """In-place assignment M1.assign(M2): copy values of M2 (matrix or list of lists)
        into M1. Only allowed if the shapes agree."""
        new = self._validated_copy(other)
        if (len(new), len(new[0])) != self.shape():
            raise ValueError(f"shape mismatch: {self.shape()} vs {(len(new), len(new[0]))}")
        for i, row in enumerate(new):
            self.data[i][:] = row
        return self

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        return matrix([list(col) for col in zip(*self.data)])

    def row(self, n):
        return matrix([self.data[n][:]])

    def column(self, n):
        return matrix([[r[n]] for r in self.data])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        """Sub-matrix of rows n_0..n_1-1 and columns m_0..m_1-1 (end-exclusive, like slices)."""
        n, m = self.shape()
        if not (0 <= n_0 < n_1 <= n and 0 <= m_0 < m_1 <= m):
            raise IndexError("block indices out of range")
        return matrix([row[m_0:m_1] for row in self.data[n_0:n_1]])

    def scalarmul(self, c):
        if not self._is_number(c):
            raise TypeError("scalar must be a number")
        return matrix([[c * v for v in row] for row in self.data])

    def _check_same_shape(self, N, op):
        if not isinstance(N, matrix):
            raise TypeError(f"{op} requires a matrix")
        if self.shape() != N.shape():
            raise ValueError(f"{op}: shape mismatch {self.shape()} vs {N.shape()}")

    def add(self, N):
        self._check_same_shape(N, "add")
        return matrix([[a + b for a, b in zip(r1, r2)] for r1, r2 in zip(self.data, N.data)])

    def sub(self, N):
        self._check_same_shape(N, "sub")
        return matrix([[a - b for a, b in zip(r1, r2)] for r1, r2 in zip(self.data, N.data)])

    def mat_mult(self, N):
        if not isinstance(N, matrix):
            raise TypeError("mat_mult requires a matrix")
        n, k = self.shape()
        k2, m = N.shape()
        if k != k2:
            raise ValueError(f"mat_mult: inner dimensions differ ({k} vs {k2})")
        cols = N.transpose().data
        return matrix([[sum(a * b for a, b in zip(row, col)) for col in cols] for row in self.data])

    def element_mult(self, N):
        self._check_same_shape(N, "element_mult")
        return matrix([[a * b for a, b in zip(r1, r2)] for r1, r2 in zip(self.data, N.data)])

    def equals(self, N):
        return isinstance(N, matrix) and self.shape() == N.shape() and self.data == N.data

    def __mul__(self, other):
        if isinstance(other, matrix):
            return self.mat_mult(other)
        if self._is_number(other):
            return self.scalarmul(other)
        return NotImplemented

    def __rmul__(self, other):
        if self._is_number(other):
            return self.scalarmul(other)
        return NotImplemented

    def __add__(self, other):
        return self.add(other) if isinstance(other, matrix) else NotImplemented

    def __sub__(self, other):
        return self.sub(other) if isinstance(other, matrix) else NotImplemented

    def __eq__(self, other):
        return self.equals(other) if isinstance(other, matrix) else NotImplemented

    __hash__ = None

    def __repr__(self):
        return f"matrix({self.data})"

    def __str__(self):
        cells = [[str(v) for v in row] for row in self.data]
        w = max(len(s) for row in cells for s in row)
        return "\n".join("[ " + "  ".join(s.rjust(w) for s in row) + " ]" for row in cells)

**Tests for part 1.** (Python's plain `M_1 = M_2` only rebinds a name and cannot be overloaded, so value assignment is provided by `M_1.assign(M_2)` and the equivalent `M_1[:] = M_2`; both accept a `matrix` or a list of lists and raise an error on a size mismatch.)

In [2]:
def raises(exc, f):
    """Return True if calling f() raises exc."""
    try:
        f()
    except exc:
        return True
    return False

Z = matrix(2, 3)
print(Z)
assert Z.to_list() == [[0, 0, 0], [0, 0, 0]] and Z.shape() == (2, 3)

M = matrix([[1, 2, 3], [4, 5, 6]])
print(M)
assert M.shape() == (2, 3)

assert raises(ValueError, lambda: matrix([[1, 2], [3]]))
assert raises(ValueError, lambda: matrix([]))
assert raises(ValueError, lambda: matrix([[]]))
assert raises(TypeError,  lambda: matrix([1, 2, 3]))
assert raises(TypeError,  lambda: matrix([[1, "a"], [3, 4]]))
assert raises(ValueError, lambda: matrix(0, 3))
assert raises(TypeError,  lambda: matrix("hello"))
print("initialisation + validation tests passed")

assert M[1][2] == 6 and M[1, 2] == 6 and M[0][0] == 1 and M[0, 0] == 1
M[0][1] = 20
assert M[0, 1] == 20
M[0, 1] = 2
assert M[0][1] == 2
print("indexing tests passed")

A = matrix(2, 3)
A.assign(M)
assert A == M and A is not M
A[:] = matrix([[9, 9, 9], [8, 8, 8]])
assert A.to_list() == [[9, 9, 9], [8, 8, 8]]
assert M.to_list() == [[1, 2, 3], [4, 5, 6]]
assert raises(ValueError, lambda: A.assign(matrix(3, 2)))

A.assign([[1, 1, 1], [2, 2, 2]])
assert A.to_list() == [[1, 1, 1], [2, 2, 2]]
A[:] = [[0, 1, 0], [1, 0, 1]]
assert A.to_list() == [[0, 1, 0], [1, 0, 1]]
assert raises(ValueError, lambda: A.assign([[1, 2], [3, 4]]))
assert raises(ValueError, lambda: A.assign([[1, 2, 3], [4, 5]]))
print("assignment tests passed")

[ 0  0  0 ]
[ 0  0  0 ]
[ 1  2  3 ]
[ 4  5  6 ]
initialisation + validation tests passed
indexing tests passed
assignment tests passed


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows.
    * Modify `__getitem__` implemented above to support slicing.
        

In [3]:
M = matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
R = matrix([[1, 2, 3, 4], [5, 6, 7, 8]])

assert M.shape() == (3, 3) and R.shape() == (2, 4)

T = R.transpose()
print(T)
assert T.to_list() == [[1, 5], [2, 6], [3, 7], [4, 8]] and T.shape() == (4, 2)
assert R.shape() == (2, 4) and T.transpose() == R

assert M.row(1).to_list() == [[4, 5, 6]] and M.row(1).shape() == (1, 3)
assert M.column(2).to_list() == [[3], [6], [9]] and M.column(2).shape() == (3, 1)

L = M.to_list()
L[0][0] = 99
assert M[0, 0] == 1

B = M.block(0, 2, 1, 3)
print(B)
assert B.to_list() == [[2, 3], [5, 6]]
assert M.block(1, 2, 0, 3) == M.row(1)
assert raises(IndexError, lambda: M.block(0, 4, 0, 1))
assert raises(IndexError, lambda: M.block(2, 1, 0, 1))

assert M[0:2].to_list() == [[1, 2, 3], [4, 5, 6]]
assert M[:, 1].to_list() == [[2], [5], [8]]
assert M[1, :].to_list() == [[4, 5, 6]]
assert M[0:2, 1:3].to_list() == [[2, 3], [5, 6]]
assert M[::2, ::2].to_list() == [[1, 3], [7, 9]]
assert M[-1, -1] == 9 and M[-1][0] == 7
assert M[1] == [4, 5, 6] and M[1][1] == 5
print("method + slicing tests passed")

[ 1  5 ]
[ 2  6 ]
[ 3  7 ]
[ 4  8 ]
[ 2  3 ]
[ 5  6 ]
method + slicing tests passed


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [4]:
def constant(n, m, c):
    """n x m matrix filled with the float c."""
    return matrix([[float(c)] * m for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0)

def ones(n, m):
    return constant(n, m, 1)

def eye(n):
    """n x n identity matrix (floats)."""
    return matrix([[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)])

In [5]:
c = constant(2, 3, 7)
print(c)
assert c.shape() == (2, 3) and all(v == 7.0 and isinstance(v, float) for row in c.to_list() for v in row)

z = zeros(3, 2)
assert z.shape() == (3, 2) and z.to_list() == [[0.0, 0.0]] * 3
assert all(isinstance(v, float) for row in z.to_list() for v in row)

o = ones(2, 2)
assert o.to_list() == [[1.0, 1.0], [1.0, 1.0]]

I = eye(3)
print(I)
assert I.to_list() == [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
assert eye(1).to_list() == [[1.0]]
print("special matrix tests passed")

[ 7.0  7.0  7.0 ]
[ 7.0  7.0  7.0 ]
[ 1.0  0.0  0.0 ]
[ 0.0  1.0  0.0 ]
[ 0.0  0.0  1.0 ]
special matrix tests passed


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [6]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])
R = matrix([[1, 2, 3], [4, 5, 6]])
S = matrix([[1, 2], [3, 4], [5, 6]])

assert A.scalarmul(3).to_list() == [[3, 6], [9, 12]]
assert A.scalarmul(0.5).to_list() == [[0.5, 1.0], [1.5, 2.0]]
assert A.to_list() == [[1, 2], [3, 4]]
assert raises(TypeError, lambda: A.scalarmul("a"))

assert A.add(B).to_list() == [[6, 8], [10, 12]]
assert B.sub(A).to_list() == [[4, 4], [4, 4]]
assert raises(ValueError, lambda: A.add(R))
assert raises(ValueError, lambda: A.sub(R))
assert raises(TypeError, lambda: A.add(5))

assert A.mat_mult(B).to_list() == [[19, 22], [43, 50]]
P = R.mat_mult(S)
assert P.shape() == (2, 2) and P.to_list() == [[22, 28], [49, 64]]
Q = S.mat_mult(R)
assert Q.shape() == (3, 3)
assert raises(ValueError, lambda: R.mat_mult(R))

assert A.element_mult(B).to_list() == [[5, 12], [21, 32]]
assert raises(ValueError, lambda: A.element_mult(R))

assert A.equals(matrix([[1, 2], [3, 4]]))
assert not A.equals(B)
assert not A.equals(R)
assert not A.equals([[1, 2], [3, 4]])
print("arithmetic tests passed")

arithmetic tests passed


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


In [7]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])

print("2*A =");  print(2 * A)
print("A*2 =");  print(A * 2)
print("A+B =");  print(A + B)
print("A-B =");  print(A - B)
print("A*B =");  print(A * B)
print("A==B:", A == B, "  A==A copy:", A == matrix(A.to_list()))

assert (2 * A) == A.scalarmul(2) == (A * 2)
assert (A + B) == A.add(B)
assert (A - B) == A.sub(B)
assert (A * B) == A.mat_mult(B)
assert (A == B) is False and (A == matrix(A.to_list())) is True
assert (A != B) is True
assert (A == 5) is False
assert raises(ValueError, lambda: A + matrix(3, 3))
assert raises(ValueError, lambda: A * matrix(3, 3))
assert raises(TypeError, lambda: A + 1)
assert (2 * A + B * 3 - A) == matrix([[16, 20], [24, 28]])
print("operator tests passed")

2*A =
[ 2  4 ]
[ 6  8 ]
A*2 =
[ 2  4 ]
[ 6  8 ]
A+B =
[  6   8 ]
[ 10  12 ]
A-B =
[ -4  -4 ]
[ -4  -4 ]
A*B =
[ 19  22 ]
[ 43  50 ]
A==B: False   A==A copy: True
operator tests passed


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [8]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[0, 1], [1, 0]])
C = matrix([[2, -1], [5, 3]])
I = eye(2)

lhs, rhs = (A * B) * C, A * (B * C)
print("(AB)C =");  print(lhs)
print("A(BC) =");  print(rhs)
print("equal:", lhs == rhs)
assert lhs == rhs

lhs, rhs = A * (B + C), A * B + A * C
print("\nA(B+C) =");  print(lhs)
print("AB + AC =");   print(rhs)
print("equal:", lhs == rhs)
assert lhs == rhs

print("\nAB =");  print(A * B)
print("BA =");    print(B * A)
print("equal:", A * B == B * A)
assert A * B != B * A

print("\nAI =");  print(A * I)
print("A  =");   print(A)
print("equal:", A * I == A)
assert A * I == A and I * A == A

(AB)C =
[  9   1 ]
[ 23   5 ]
A(BC) =
[  9   1 ]
[ 23   5 ]
equal: True

A(B+C) =
[ 14   6 ]
[ 30  12 ]
AB + AC =
[ 14   6 ]
[ 30  12 ]
equal: True

AB =
[ 2  1 ]
[ 4  3 ]
BA =
[ 3  4 ]
[ 1  2 ]
equal: False

AI =
[ 1.0  2.0 ]
[ 3.0  4.0 ]
A  =
[ 1  2 ]
[ 3  4 ]
equal: True
